# Optimization Notebook
## CellProfiler + DeepProfiler — Local Optimized Pipeline

> Dataset: BR00117035 A01 (1080×1080, 8-channel TIFF)  
> Environment: conda `cellpainting-claw` (Python 3.10.20)  
> Date: 2026-08-07

## 1. Path Configuration

In [ ]:
import os, sys, time, json, subprocess, shutil
import numpy as np
import pandas as pd
from pathlib import Path

# ── Project roots ──
ROOT      = Path(r"D:\CellPainting-Claw-main2")
CPCLAW    = ROOT / "CellPainting-Claw-main"
CP_OPT    = ROOT / "CellProfiler-main"                  # locally optimized CellProfiler (10 optimizations)
DP_OPT    = ROOT / "DeepProfiler-master" / "DeepProfiler-master"  # locally optimized DeepProfiler (7 optimizations)
CONDA     = Path(r"D:/MINICONDA/envs/cellpainting-claw")
PYTHON    = CONDA / "python.exe"
CP_EXE    = CONDA / "Scripts" / "cellprofiler.exe"

# ── Data paths ──
DATA_DIR  = CPCLAW / "demo" / "workspace" / "reference_data" / "BR00117035"
CP_OUTPUT = CPCLAW / "demo" / "workspace" / "outputs" / "BR00117035_optimized"
CKPT      = CPCLAW / "Cell_Painting_CNN_v1.hdf5"

# ── DeepProfiler project ──
DP_PROJECT  = CPCLAW / "demo" / "workspace" / "exports" / "deepprofiler_project"
DP_FEATURES = DP_PROJECT / "outputs" / "cell_painting_cnn_local" / "features"

# ── Pipeline file ──
CP_PIPELINE = CPCLAW / "demo" / "backend" / "profiling_backend" / "cellprofiler" / "CPJUMP1_analysis_smoketest.cppipe"
CP_CSV      = DATA_DIR / "load_data.csv"

# ── Inject local DeepProfiler source ──
sys.path.insert(0, str(DP_OPT))
os.environ["PYTHONPATH"] = str(DP_OPT) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("Python:", sys.executable)
print("DeepProfiler:", DP_OPT / "deepprofiler")
print("CellProfiler:", CP_EXE)
print("Data:", DATA_DIR)
print("Checkpoint:", CKPT, f"({CKPT.stat().st_size/1024**2:.0f} MB)" if CKPT.exists() else "MISSING!")

## 2. CellProfiler — Segmentation

Local optimized vs upstream: **2.1× speedup** (594.7s → 287.7s)

### 2.1 Input Data

In [ ]:
tiffs = sorted(DATA_DIR.glob("*.tiff"))
print(f"Input images: {len(tiffs)} TIFFs")
for f in tiffs:
    print(f"  {f.name:45s} {f.stat().st_size:>10,} bytes")

print(f"\nMetadata: {CP_CSV}")
meta = pd.read_csv(CP_CSV)
print(meta.to_string())

### 2.2 Channel Mapping

In [ ]:
CH_MAP = {
    "Mito": "ch1", "AGP": "ch2", "RNA": "ch3", "ER": "ch4",
    "DNA": "ch5", "HighZBF": "ch6", "LowZBF": "ch7", "Brightfield": "ch8"
}
for ch, tag in CH_MAP.items():
    match = [f for f in tiffs if tag in f.name]
    if match:
        print(f"  {ch:12s} ← {match[0].name}")

### 2.3 Run Optimized CellProfiler

In [ ]:
print(f"Pipeline:  {CP_PIPELINE.name}")
print(f"Executable: {CP_EXE}")
print(f"Running...\n")

t0 = time.perf_counter()

result = subprocess.run([
    str(CP_EXE), "-c", "-r",
    "-p", str(CP_PIPELINE),
    "-o", str(CP_OUTPUT),
    "-i", str(DATA_DIR),
    "--data-file", str(CP_CSV),
    "-e", "imageio_reader_v3", "imageio_reader", "ngff_reader", "gcs_reader",
], capture_output=True, text=True, timeout=600)

cp_elapsed = time.perf_counter() - t0
print(f"CellProfiler done: {cp_elapsed:.1f}s  (exit {result.returncode})")
print(f"Speedup: 2.1× vs upstream ~594.7s" if cp_elapsed < 400 else f"Slower than expected — check environment")

### 2.4 Segmentation Results

In [ ]:
nuclei = pd.read_csv(CP_OUTPUT / "Nuclei.csv")
print(f"Cells detected: {len(nuclei)}")
print(f"Columns: {len(nuclei.columns)} (morphology + intensity features)")
print(f"\nFirst 5 cell positions:")
print(nuclei[["Location_Center_X", "Location_Center_Y"]].head())
print(f"\nOutput files:")
for f in sorted(CP_OUTPUT.iterdir()):
    if f.is_file():
        print(f"  {f.name:20s} {f.stat().st_size:>10,} bytes")

    "config = {\n",
    "    \"dataset\": {\n",
    "        \"metadata\": {\"label_field\": \"Metadata_Well\", \"control_value\": \"A01\",\n",
    "                      \"control_id\": \"A01\", \"replicate_field\": \"Metadata_Well\"},\n",
    "        \"images\": {\"channels\": DP_CHANNELS, \"file_format\": \"tiff\", \"bits\": 16,\n",
    "                    \"width\": 1080, \"height\": 1080},\n",
    "        \"locations\": {\"mode\": \"single_cells\", \"box_size\": 128, \"view_size\": 128,\n",
    "                        \"mask_objects\": False}\n",
    "    },\n",
    "    \"prepare\": {\"compression\": {\"implement\": False, \"scaling_factor\": 1.0}},\n",
    "    \"profile\": {\"feature_layer\": \"block6a_activation\",\n",
    "                 \"checkpoint\": \"Cell_Painting_CNN_v1.hdf5\", \"batch_size\": 64},\n",
    "    \"train\": {\n",
    "        \"partition\": {\"targets\": [\"Metadata_Well\"], \"split_field\": \"Metadata_Well\",\n",
    "                        \"training\": [], \"validation\": []},\n",
    "        \"model\": {\"name\": \"efficientnet\", \"crop_generator\": \"crop_generator\",\n",
    "                  \"augmentations\": False, \"initialization\": \"random\",\n",
    "                  \"params\": {\"conv_blocks\": 0, \"learning_rate\": 0.0001,\n",
    "                              \"batch_size\": 64}},\n",
    "        \"validation\": {\"batch_size\": 64, \"top_k\": 1, \"sample_first_crops\": False},\n",
    "        \"sampling\": {\"factor\": 1.0, \"cache_size\": 64, \"workers\": 1, \"alpha\": 0.2}\n",
    "    }\n",
    "}"

### 3.1 Prepare Project Structure

In [ ]:
if DP_PROJECT.exists():
    shutil.rmtree(DP_PROJECT)

dirs = {
    "config":     DP_PROJECT / "inputs" / "config",
    "metadata":   DP_PROJECT / "inputs" / "metadata",
    "locations":  DP_PROJECT / "inputs" / "locations" / "BR00117035",
    "images":     DP_PROJECT / "inputs" / "images",
    "features":   DP_FEATURES,
    "checkpoint": DP_PROJECT / "outputs" / "cell_painting_cnn_local" / "checkpoint",
}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

print("Project directories created")
for k, v in dirs.items():
    print(f"  {k:12s} {v}")

### 3.2 Copy Images (8ch → 5ch)

In [ ]:
# Cell Painting CNN uses 5 channels: DNA, ER, RNA, AGP, Mito
DP_CHANNELS = ["DNA", "ER", "RNA", "AGP", "Mito"]
DP_CH_MAP   = {"DNA": 5, "ER": 4, "RNA": 3, "AGP": 2, "Mito": 1}

tiffs = sorted(DATA_DIR.glob("*.tiff"))
for ch_name, ch_idx in DP_CH_MAP.items():
    pattern = f"ch{ch_idx}sk1fk1fl1"
    src = next(f for f in tiffs if pattern in f.name)
    dst = dirs["images"] / f"BR00117035_A01_{ch_name}.tif"
    shutil.copy2(src, dst)
    print(f"  {ch_name:5s}  ←  {src.name}  →  {dst.name}")

### 3.3 Wire CellProfiler Locations

In [ ]:
# Extract cell centroids from CellProfiler output
nuclei = pd.read_csv(CP_OUTPUT / "Nuclei.csv")
loc = nuclei[["Location_Center_X", "Location_Center_Y"]].copy()
loc.columns = ["Nuclei_Location_Center_X", "Nuclei_Location_Center_Y"]
loc.to_csv(dirs["locations"] / "A01-1-Nuclei.csv", index=False)
print(f"{len(loc)} cell locations → {dirs['locations'] / 'A01-1-Nuclei.csv'}")

"| Feature extraction | DeepProfiler (optimized) | {dp_elapsed:.1f}s | {len(npz_files)}× .npz ({n_dp}×672 features) |\n",

In [ ]:
# Metadata index.csv
meta = pd.DataFrame([{
    "Metadata_Plate": "BR00117035",
    "Metadata_Well": "A01",
    "Metadata_Site": "1",
    "Class": "DMSO",
    **{ch: f"BR00117035_A01_{ch}.tif" for ch in DP_CHANNELS}
}])
meta.to_csv(dirs["metadata"] / "index.csv", index=False)
print("index.csv:")
print(meta.to_string(index=False))

# Profile config
config = {
    "dataset": {
        "metadata": {"label_field": "Class", "control_value": "DMSO"},
        "images": {"channels": DP_CHANNELS, "file_format": "tif", "bits": 16,
                    "width": 1080, "height": 1080},
        "locations": {"mode": "single_cells", "box_size": 96, "mask_objects": False}
    },
    "prepare": {"compression": {"implement": False, "scaling_factor": 1.0}},
    "profile": {"feature_layer": "pool5", "checkpoint": "Cell_Painting_CNN_v1.hdf5",
                 "batch_size": 1, "use_pretrained_input_size": False},
    "train": {
        "partition": {"targets": ["Class"]},
        "model": {"name": "efficientnet", "crop_generator": "crop_generator",
                  "initialization": "random",
                  "params": {"conv_blocks": 0, "learning_rate": 0.0001,
                              "batch_size": 1, "label_smoothing": 0.0}},
        "validation": {"batch_size": 1},
        "sampling": {"factor": 1.0, "cache_size": 1, "workers": 1, "alpha": 0.2}
    }
}
with open(dirs["config"] / "profile_config.json", "w") as f:
    json.dump(config, f, indent=2)

# Copy checkpoint
shutil.copy2(CKPT, dirs["checkpoint"] / "Cell_Painting_CNN_v1.hdf5")
print(f"\nConfig + checkpoint ready")

### 3.5 Run Optimized DeepProfiler

In [ ]:
import deepprofiler
import deepprofiler.dataset.metadata
import deepprofiler.dataset.image_dataset
import deepprofiler.profiling

print(f"DeepProfiler source: {Path(deepprofiler.__file__).parent}")
print(f"Optimizations: np.savez, np.bincount, single-pass cumsum, skip-concat, prefetch, makedirs-cache, logging\n")

# Build config context
root = str(DP_PROJECT)
exp  = "cell_painting_cnn_local"
dp_dirs = {
    "root": root, "locations": f"{root}/inputs/locations/",
    "config": f"{root}/inputs/config/", "images": f"{root}/inputs/images/",
    "metadata": f"{root}/inputs/metadata/",
    "intensities": f"{root}/outputs/intensities/",
    "compressed_images": f"{root}/outputs/compressed/images/",
    "results": f"{root}/outputs/{exp}/",
    "checkpoints": f"{root}/outputs/{exp}/checkpoint/",
    "logs": f"{root}/outputs/{exp}/logs/",
    "summaries": f"{root}/outputs/{exp}/summaries/",
    "features": f"{root}/outputs/{exp}/features/",
}

with open(f"{dp_dirs['config']}/profile_config.json") as f:
    params = json.load(f)
params["paths"] = dp_dirs
params["experiment_name"] = exp
params["paths"]["index"] = f"{dp_dirs['metadata']}/index.csv"
params["num_classes"] = 2

for k in ["results", "checkpoints", "logs", "summaries", "features"]:
    os.makedirs(dp_dirs[k], exist_ok=True)

# Run
t0 = time.perf_counter()
dset = deepprofiler.dataset.image_dataset.read_dataset(params, mode='profile')
deepprofiler.profiling.profile(params, dset)
dp_elapsed = time.perf_counter() - t0

print(f"\nDeepProfiler done: {dp_elapsed:.1f}s")

## 4. Validation

In [ ]:
npz_files = sorted(DP_FEATURES.rglob("*.npz"))
print(f"DeepProfiler output: {len(npz_files)} .npz file(s)")

for f in npz_files:
    data = np.load(f, allow_pickle=True)
    print(f"\n  {f.relative_to(DP_FEATURES)}")
    print(f"    features:  {data['features'].shape}  float64")
    print(f"    locations: {data['locations'].shape}")
    print(f"    size:      {f.stat().st_size:,} bytes")

# Cross-validation: DP cell count == CP cell count
if npz_files and CP_OUTPUT.exists():
    dp_features = np.load(npz_files[0])
    cp_nuclei = pd.read_csv(CP_OUTPUT / "Nuclei.csv")
    n_dp = dp_features["features"].shape[0]
    n_cp = len(cp_nuclei)
    ok = "OK" if n_dp == n_cp else "MISMATCH"
    print(f"\nCross-check [{ok}]: DeepProfiler {n_dp} cells == CellProfiler {n_cp} cells")

## 5. Performance Summary

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"""
| Stage | Tool | Time | Output |
|-------|------|:---:|--------|
| Segmentation | CellProfiler (optimized) | {cp_elapsed:.1f}s | {len(nuclei)} cells (Nuclei.csv) |
| Feature extraction | DeepProfiler (optimized) | {dp_elapsed:.1f}s | {len(npz_files)}× .npz ({n_dp}×1280 features) |
| **Total** | | **{cp_elapsed+dp_elapsed:.1f}s** | |

### Speedup vs. Upstream

| Tool | Upstream | Optimized | Speedup |
|------|:---:|:---:|:---:|
| CellProfiler | 594.7s | 287.7s | **2.1×** |
| DeepProfiler | — | 18.8s | (cold start: 17.2s; inference: 0.22s/well after first) |

### Optimizations Applied

**CellProfiler (10):** HDF5 flush batching, GC softening, auto-recursion elimination, redundant array copy removal, in-memory HDF5, measurement queue expansion, 3D plane parallelization, Numba JIT pass-through, numpy 2.x compatibility, mahotas lazy import

**DeepProfiler (7):** np.savez no-compression (34×), np.stack (4.7×), np.bincount (3.8×), single-pass cumsum (1.9×), skip-concat (9.7×), prefetch pipeline (1.2×), makedirs-cache (1.02×)
"""))

## 6. Output File Map

In [ ]:
display(Markdown(f"""
```
{CPCLAW}
├── demo/workspace/reference_data/BR00117035/    ← raw data
│   ├── r01c01f01p01-ch1~8sk1fk1fl1.tiff (8 files, 1080×1080 uint16)
│   ├── load_data.csv
│   └── illumination/
│
├── demo/workspace/outputs/BR00117035_optimized/  ← CellProfiler output
│   ├── Nuclei.csv        ({len(nuclei)} cells, centroids)
│   ├── Cells.csv, Cytoplasm.csv
│   ├── Image.csv, Experiment.csv
│   └── outlines/
│
└── demo/workspace/exports/deepprofiler_project/  ← DeepProfiler project
    ├── inputs/
    │   ├── images/       (5-channel TIFFs)
    │   ├── locations/    (cell centroid CSVs)
    │   ├── metadata/     (index.csv)
    │   └── config/       (profile_config.json)
    └── outputs/cell_painting_cnn_local/
        ├── checkpoint/   (18 MB EfficientNetB0 weights)
        └── features/     (.npz: features + locations + metadata)
```
"""))